In [ ]:
import matplotlib
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.utils import resample
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
# import cupy
import os
from sklearn.model_selection import cross_val_score
from dotenv import load_dotenv, dotenv_values 
# loading variables from .env file
load_dotenv() 

import mlflow

mlflow.set_tracking_uri(os.getenv('MLFLOW_SERVER'))
mlflow.set_experiment("Kaggle-s6e4")

print('mlflow server: ', os.getenv('MLFLOW_SERVER'))
kaggle = True if os.environ.get('KAGGLE_URL_BASE','') else False
selected_features = False
inliers = False

if kaggle:
    training_data = '/kaggle/input/competitions/playground-series-s6e4/train.csv'
else:
    training_data = 'data/train.csv'

df_tv = pd.read_csv(training_data)
df_x = df_tv.iloc[:,1:-1]

df_dummy = pd.get_dummies(df_x, dtype=int, drop_first=False)

if selected_features:
    df_label = df_tv.replace({'Irrigation_Need': {'Low': 0, 'Medium': 1, 'High': 2}})['Irrigation_Need']
    df_full = pd.concat([df_dummy, df_label], axis=1)
    corr = df_full.corr()
    df_corr_sort_abs = corr.abs().sort_values(by='Irrigation_Need', ascending=False)['Irrigation_Need']
    threshold = 0.1
    df_dummy = df_full[df_corr_sort_abs[df_corr_sort_abs > threshold].index]
    df_dummy.drop(columns=['Irrigation_Need'], inplace=True)

x = df_dummy.iloc[:,:].values
y = df_tv.iloc[:,-1].values

class_le = LabelEncoder()
y = class_le.fit_transform(y)

In [ ]:
# Enable autologging for scikit-learn
mlflow.sklearn.autolog()
mlflow.lightgbm.autolog()

In [ ]:

from lightgbm import LGBMClassifier

# device='cuda' is only supported in lightgbm built with: 
# pip install lightgbm --install-option=cmake.define.USE_CUDA=ON
lgbm = LGBMClassifier(class_weight="balanced", n_estimators=500, 
                    learning_rate=0.1, max_depth=3, reg_lambda=1,
                    boosting_type='gbdt', is_unbalance=True,objective='multiclass', 
                    random_state=1, verbose=-1)
                    #device='cuda')
lgbm.fit(x, y)
#cv_scores = cross_val_score(lgbm, x, y, cv=2, scoring='balanced_accuracy', error_score='raise')
#print(f"Mean cross-validation score: {np.mean(cv_scores):.3f}")

LightGBMError: No OpenCL device found

In [ ]:
import optuna
import sklearn
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import make_scorer, balanced_accuracy_score
from lightgbm import LGBMClassifier
mlflow.set_experiment("S6E4: Hyperparameter Tuning Experiment")

def objective(trial):
    # Setting nested=True will create a child run under the parent run.
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as child_run:
        lgb_max_depth = trial.suggest_int("lgb_max_depth", 2, 10)
        lgb_n_estimators = trial.suggest_int("lgb_n_estimators", 100, 10000, step=100)
        lgb_learning_rate = trial.suggest_float("lgb_learning_rate", 0.01, 1.0)
        lgb_l2_regularization = trial.suggest_float("lgb_l2_regularization", 0.0, 100.0)
        lgb_max_bins = trial.suggest_int("lgb_max_bins", 2, 255)
        lgb_reg_alpha = trial.suggest_float("lgb_reg_alpha", 0.0, 100.0)
        lgb_min_data_in_leaf = trial.suggest_int("lgb_min_data_in_leaf", 1, 100)
        params = {
            "max_depth": lgb_max_depth,
            "class_weight": 'balanced',
            "n_estimators": lgb_n_estimators,
            "learning_rate": lgb_learning_rate,
            "reg_lambda": lgb_l2_regularization,
            "max_bin": lgb_max_bins,
            "reg_alpha": lgb_reg_alpha,
            "min_data_in_leaf": lgb_min_data_in_leaf,
            "objective": 'multiclass',
            'boosting_type': 'gbdt',
            #"device": 'cuda',
            "random_state": 1

        }
        # Log current trial's parameters
        mlflow.log_params(params)

        regressor_obj = LGBMClassifier(**params)

        #kf = StratifiedKFold(n_splits=2, shuffle=True, random_state=1)
        balanced_accuracy_scorer = make_scorer(balanced_accuracy_score)
        scores = cross_val_score(regressor_obj, x, y, scoring='balanced_accuracy', cv=5)
        
        # regressor_obj.fit(x, y)

        # y_pred = regressor_obj.predict(x)
        # error = sklearn.metrics.mean_squared_error(y, y_pred)
        # # Log current trial's error metric
        mlflow.log_metrics({"mean_score": np.mean(scores), "median_score": np.median(scores)})

        # Log the model file
        mlflow.sklearn.log_model(regressor_obj, name="model")
        # Make it easy to retrieve the best-performing child run later
        trial.set_user_attr("run_id", child_run.info.run_id)
        #return error
        return np.min([np.mean(scores), np.median(scores)])

In [ ]:
# Create a parent run that contains all child runs for different trials
with mlflow.start_run(run_name="LGBM") as run:
    # Log the experiment settings
    n_trials = 30
    mlflow.log_param("n_trials", n_trials)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    # Log the best trial and its run ID
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metrics({"best_error": study.best_value})
    if best_run_id := study.best_trial.user_attrs.get("run_id"):
        mlflow.log_param("best_child_run_id", best_run_id)

In [11]:
from catboost import CatBoostClassifier

params = {'iterations': 2000,
          'depth': 3,
          'learning_rate': .1,
          'loss_function': 'MultiClass',
          'verbose': False,
          'task_type': 'GPU',
          'reg_lambda': 0.1,
          'auto_class_weights': 'Balanced',
          'random_state': 1}
          #'class_weights': weights}
#cb = CatBoostClassifier(**params)
                        
from sklearn.base import clone

class CustomCatBoostClassifier(CatBoostClassifier):
    def __sklearn_clone__(self):
        return CustomCatBoostClassifier(**self.get_params())
    
cb = CustomCatBoostClassifier(**params)
cb.fit(x, y)
#cv_scores = cross_val_score(cb, x, y, cv=2, scoring='balanced_accuracy', error_score='raise')
#print(f"Mean cross-validation score: {np.mean(cv_scores):.3f}")

CustomCatBoostClassifier(auto_class_weights='Balanced', depth=3, iterations=2000, learning_rate=0.1, loss_function='MultiClass', random_state=1, reg_lambda=0.1, task_type='GPU', verbose=False)

In [12]:
import optuna
import sklearn
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import make_scorer, balanced_accuracy_score
from catboost import CatBoostClassifier
from sklearn.base import clone
mlflow.set_experiment("S6E4: Hyperparameter Tuning Experiment")

def objective(trial):
    # Setting nested=True will create a child run under the parent run.
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as child_run:
        cb_depth = trial.suggest_int("cb_depth", 2, 10)
        cb_iterations = trial.suggest_int("cb_iterations", 100, 10000, step=100)
        cb_learning_rate = trial.suggest_float("cb_learning_rate", 0.01, 1.0)
        cb_l2_leaf_reg = trial.suggest_float("cb_l2_leaf_reg", 0.0, 100.0)
        bagging_temperature = trial.suggest_float("bagging_temperature", 0.0, 1.0)

        params = {
            "depth": cb_depth,
            "auto_class_weights": 'Balanced',
            "iterations": cb_iterations,
            "learning_rate": cb_learning_rate,
            "l2_leaf_reg": cb_l2_leaf_reg,
            'bagging_temperature': bagging_temperature,
            'loss_function': 'MultiClass',
            'verbose': False,
            'task_type': 'GPU',
            #"device": 'cuda',
            "random_state": 1

        }
        # Log current trial's parameters
        mlflow.log_params(params)

        class CustomCatBoostClassifier(CatBoostClassifier):
            def __sklearn_clone__(self):
                return CustomCatBoostClassifier(**self.get_params())

        regressor_obj = CustomCatBoostClassifier(**params)

        #kf = StratifiedKFold(n_splits=2, shuffle=True, random_state=1)
        balanced_accuracy_scorer = make_scorer(balanced_accuracy_score)
        scores = cross_val_score(regressor_obj, x, y, scoring='balanced_accuracy', cv=5)
        
        # regressor_obj.fit(x, y)

        # y_pred = regressor_obj.predict(x)
        # error = sklearn.metrics.mean_squared_error(y, y_pred)
        # # Log current trial's error metric
        mlflow.log_metrics({"mean_score": np.mean(scores), "median_score": np.median(scores)})

        # Log the model file
        mlflow.sklearn.log_model(regressor_obj, name="model")
        # Make it easy to retrieve the best-performing child run later
        trial.set_user_attr("run_id", child_run.info.run_id)
        #return error
        return np.min([np.mean(scores), np.median(scores)])

In [ ]:
# Create a parent run that contains all child runs for different trials
with mlflow.start_run(run_name="CB") as run:
    # Log the experiment settings
    n_trials = 30
    mlflow.log_param("n_trials", n_trials)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    # Log the best trial and its run ID
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metrics({"best_error": study.best_value})
    if best_run_id := study.best_trial.user_attrs.get("run_id"):
        mlflow.log_param("best_child_run_id", best_run_id)

[I 2026-04-17 15:46:54,562] A new study created in memory with name: no-name-2c71b988-7fb2-4f75-bdf4-874c76923d88
2026/04/17 15:47:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-04-17 15:47:17,333] Trial 0 finished with value: 0.9653107021799603 and parameters: {'cb_depth': 3, 'cb_iterations': 1200, 'cb_learning_rate': 0.9934736842054008, 'cb_l2_leaf_reg': 29.182554186957056, 'bagging_temperature': 0.3665722850732316}. Best is trial 0 with value: 0.9653107021799603.


🏃 View run trial_0 at: http://192.168.2.87:5000/#/experiments/2/runs/bf0e85eecd174a31aa3f716ef113b035
🧪 View experiment at: http://192.168.2.87:5000/#/experiments/2


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(class_weight='balanced', max_iter=500, 
                                    learning_rate=0.1, max_depth=3, l2_regularization=0.1,
                                    loss='log_loss', random_state=1)
#hgb.fit(x, y)
#cv_scores = cross_val_score(hgb, x, y, cv=2, scoring='balanced_accuracy', error_score='raise')
#print(f"Mean cross-validation score: {np.mean(cv_scores):.3f}")

In [ ]:
import optuna
import sklearn
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import make_scorer, balanced_accuracy_score
from sklearn.ensemble import HistGradientBoostingClassifier
mlflow.set_experiment("S6E4: Hyperparameter Tuning Experiment")

def objective(trial):
    # Setting nested=True will create a child run under the parent run.
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as child_run:
        hb_max_depth = trial.suggest_int("hb_max_depth", 2, 10)
        hb_max_iter = trial.suggest_int("hb_max_iter", 100, 10000, step=100)
        hb_learning_rate = trial.suggest_float("hb_learning_rate", 0.01, 1.0)
        hb_l2_regularization = trial.suggest_float("hb_l2_regularization", 0.0, 100.0)
        hb_max_bins = trial.suggest_int("hb_max_bins", 2, 255)
        params = {
            "max_depth": hb_max_depth,
            "class_weight": 'balanced',
            "max_iter": hb_max_iter,
            "learning_rate": hb_learning_rate,
            "l2_regularization": hb_l2_regularization,
            "max_bins": hb_max_bins,
            "loss": 'log_loss',
            "random_state": 1

        }
        # Log current trial's parameters
        mlflow.log_params(params)

        regressor_obj = HistGradientBoostingClassifier(**params)

        #kf = StratifiedKFold(n_splits=2, shuffle=True, random_state=1)
        balanced_accuracy_scorer = make_scorer(balanced_accuracy_score)
        scores = cross_val_score(regressor_obj, x, y, scoring='balanced_accuracy', cv=5)
        
        # regressor_obj.fit(x, y)

        # y_pred = regressor_obj.predict(x)
        # error = sklearn.metrics.mean_squared_error(y, y_pred)
        # # Log current trial's error metric
        mlflow.log_metrics({"mean_score": np.mean(scores), "median_score": np.median(scores)})

        # Log the model file
        mlflow.sklearn.log_model(regressor_obj, name="model")
        # Make it easy to retrieve the best-performing child run later
        trial.set_user_attr("run_id", child_run.info.run_id)
        #return error
        return np.min([np.mean(scores), np.median(scores)])

In [ ]:
# Create a parent run that contains all child runs for different trials
with mlflow.start_run(run_name="HGB") as run:
    # Log the experiment settings
    n_trials = 30
    mlflow.log_param("n_trials", n_trials)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    # Log the best trial and its run ID
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metrics({"best_error": study.best_value})
    if best_run_id := study.best_trial.user_attrs.get("run_id"):
        mlflow.log_param("best_child_run_id", best_run_id)

In [ ]:
# from optuna.integration import OptunaSearchCV
# hb_max_features = optuna.distributions.FloatDistribution(0.2, 1.0)
# params = {
#     "max_features": hb_max_features,
# }
# optuna_search = OptunaSearchCV(HistGradientBoostingClassifier(max_depth=3,class_weight='balanced',
#                                     max_iter=500,
#                                     learning_rate=0.1,l2_regularization=0.1,
#                                     loss='log_loss',random_state=1), 
#                     params, cv=2, scoring='balanced_accuracy', n_trials=10,
#                     refit=True)
# optuna_search.fit(x, y)


In [ ]:
model_name_hgb = "BestHGB"
model_name_lgbm = "BestLGBM"
model_name_cb = "BestCB"
model_version = "latest"

# # Load the model from the Model Registry
model_uri_hgb = f"models:/{model_name_hgb}/{model_version}"
model_uri_lgbm = f"models:/{model_name_lgbm}/{model_version}"
model_uri_cb = f"models:/{model_name_cb}/{model_version}"

model_hgb = mlflow.sklearn.load_model(model_uri_hgb)
model_lgbm = mlflow.sklearn.load_model(model_uri_lgbm)
model_cb = mlflow.sklearn.load_model(model_uri_cb)

# model.fit(x, y)

In [ ]:
from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(estimators=[('hgb', model_hgb), ('lgbm', model_lgbm), ('cb', model_cb)], voting='soft')
#cv_scores = cross_val_score(voting_clf, x, y, cv=2, scoring='balanced_accuracy', error_score='raise')
#print(f"Mean cross-validation score: {np.mean(cv_scores):.3f}")
voting_clf.fit(x, y)

In [ ]:
if kaggle:
    testing_data = '/kaggle/input/competitions/playground-series-s6e4/test.csv'
else:
    testing_data = 'data/test.csv'

df_test = pd.read_csv(testing_data)

ids = df_test['id'].values

df_test_dummy = pd.get_dummies(df_test.iloc[:,1:], drop_first=False, dtype=int)
df_test_dummy = df_test_dummy[df_dummy.columns]
x_test = df_test_dummy.to_numpy()

In [ ]:

if kaggle:
    out_dir = '/kaggle/working/'
else:
    out_dir = 'data/'

df_submission_stack = pd.DataFrame({'id': ids, 'Irrigation_Need': class_le.inverse_transform(model.predict(x_test))})
df_submission_stack.to_csv(os.path.join(out_dir, 'mfflow_Test.csv'), index=False)